# COMETKiwi and xCOMET on the SATE planted errors and sentences

Runs the two COMET baselines of `docs/BASELINES.md` on a free GPU notebook. xCOMET-XL needs about 30 GB RAM or a 16 GB GPU, so **Kaggle** is recommended (Colab's free tier usually has too little RAM).

**Before you start**
1. On your PC: `python -m sps_validation.baselines export` → creates `build/baselines/items.jsonl`.
2. On huggingface.co (logged in), open both model pages and accept their licence: `Unbabel/wmt22-cometkiwi-da` and `Unbabel/XCOMET-XL`. Create a *read* token (Settings → Access Tokens).
3. Kaggle: New Notebook → File → Import notebook → this file. Settings: Accelerator **GPU P100** (or T4), Internet **on**. Add your token under Add-ons → Secrets with the name `HF_TOKEN`. Upload `items.jsonl` with **+ Add Input → Upload** (any dataset name).

`items.jsonl` contains the SPS text; it is processed on the notebook provider's servers.

In [ ]:
!git clone -q -b claude/gallant-einstein-07m6zf https://github.com/OniProgramming/sps_validation.git
%cd sps_validation
# A separate, clean Python 3.11 environment with versions COMET 2.2.7 works with (the notebook's own
# Python 3.13 packages conflict with it). Takes a few minutes: it downloads PyTorch with CUDA.
!pip -q install uv
!uv venv -q /kaggle/working/cenv --python 3.11
!uv pip install -q --python /kaggle/working/cenv/bin/python unbabel-comet==2.2.7 torch==2.4.1 transformers==4.44.2 pytorch-lightning==2.4.0 numpy==1.26.4 scipy==1.13.1 huggingface-hub==0.24.7 "setuptools<81"
!/kaggle/working/cenv/bin/python -c "import comet, torch; print('COMET OK, GPU:', torch.cuda.is_available())"

In [ ]:
# Hugging Face token: Kaggle secret HF_TOKEN, otherwise an interactive login (Colab)
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("token from Kaggle secrets")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# items.jsonl → build/baselines/
import glob, os, shutil
os.makedirs("build/baselines", exist_ok=True)
found = glob.glob("/kaggle/input/**/items.jsonl", recursive=True)
if found:
    shutil.copy(found[0], "build/baselines/items.jsonl")
else:  # Colab
    from google.colab import files
    up = files.upload()
    shutil.move(next(iter(up)), "build/baselines/items.jsonl")
print(sum(1 for _ in open("build/baselines/items.jsonl", encoding="utf-8")), "texts")

In [ ]:
# COMETKiwi (2.2 GB): a few minutes on a GPU
!/kaggle/working/cenv/bin/python -m sps_validation.baselines comet cometkiwi --set all --gpus 1 --batch 16

In [ ]:
# xCOMET-XL (14 GB). If you get "CUDA out of memory", add --half and run the cell again (it resumes).
!/kaggle/working/cenv/bin/python -m sps_validation.baselines comet xcomet --set all --gpus 1 --batch 4

In [ ]:
# Download the scores and put them on your PC in build/baselines/scores/
import shutil
shutil.make_archive("/kaggle/working/comet_scores" if os.path.isdir("/kaggle/working") else "comet_scores", "zip", "build/baselines/scores")
try:
    from google.colab import files
    files.download("comet_scores.zip")
except Exception:
    print("Kaggle: download comet_scores.zip from the Output panel (right side) of this notebook")